In [1]:
import pandas as pd

books = pd.read_csv("../data/books_with_categories.csv")

In [2]:
from transformers import pipeline

# classifier for Eckman 6: anger, disgust, fear, joy, neutral, sadness, surprise
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "mps")
classifier("I love this!")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528684265911579},
  {'label': 'neutral', 'score': 0.005764600355178118},
  {'label': 'anger', 'score': 0.004419781267642975},
  {'label': 'sadness', 'score': 0.002092392183840275},
  {'label': 'disgust', 'score': 0.001611993182450533},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [3]:
books["description"][0]

'Action! Adventure! Space Opera! Life hardly ever turns out the way we expect it to, and for Mykl d’Angelo, skipper and owner of the loderunner Pegasus, it had just taken a bad turn for terrible. Due a minor misunderstanding, his crew had stolen the ship’s only shuttle, leaving him and two others behind to crew the ailing ship on their own. As if that weren’t bad enough, just a few hours later the rickety old ship’s stardrive exploded in the middle of the middle of nowhere, killing Mykl’s two remaining crewmen. Marooned alone in deep space, Mykl d’Angelo counted his blessings, offered prayers to any gods who specialized in miracles, and prepared to await (A) seemingly unlikely rescue, or (B) a lingering death… Rescue takes place, but at a price – as the Antares, a cruiser dispatched to investigate the mysterious silence of a remote starbase crosses paths with a legendary and fearsome Corsair – a man whose name sent shivers down the spines of lesser mortals. Blachart… Blachart the Blood

In [4]:
classifier(books["description"][0])
# not a very accurate classification

[[{'label': 'fear', 'score': 0.7806273102760315},
  {'label': 'sadness', 'score': 0.06796623766422272},
  {'label': 'disgust', 'score': 0.06335310637950897},
  {'label': 'anger', 'score': 0.03837128356099129},
  {'label': 'neutral', 'score': 0.027229905128479004},
  {'label': 'surprise', 'score': 0.02029595337808132},
  {'label': 'joy', 'score': 0.0021562527399510145}]]

In [5]:
# classify sentences individually
classifier(books["description"][0].split("."))

[[{'label': 'fear', 'score': 0.7942874431610107},
  {'label': 'disgust', 'score': 0.06771168857812881},
  {'label': 'sadness', 'score': 0.0398513525724411},
  {'label': 'surprise', 'score': 0.03223242983222008},
  {'label': 'neutral', 'score': 0.03175150230526924},
  {'label': 'anger', 'score': 0.02174588292837143},
  {'label': 'joy', 'score': 0.012419751845300198}],
 [{'label': 'anger', 'score': 0.4011019170284271},
  {'label': 'sadness', 'score': 0.3056371212005615},
  {'label': 'neutral', 'score': 0.18285048007965088},
  {'label': 'disgust', 'score': 0.08870737999677658},
  {'label': 'fear', 'score': 0.010378353297710419},
  {'label': 'surprise', 'score': 0.009017867036163807},
  {'label': 'joy', 'score': 0.0023069221060723066}],
 [{'label': 'surprise', 'score': 0.36239781975746155},
  {'label': 'sadness', 'score': 0.1643228530883789},
  {'label': 'anger', 'score': 0.15371713042259216},
  {'label': 'neutral', 'score': 0.12592284381389618},
  {'label': 'fear', 'score': 0.096349835395

In [6]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
sorted(predictions[0], key=lambda x: x["label"])
# take the sentence with the highest probability for each sentiment

[{'label': 'anger', 'score': 0.02174588292837143},
 {'label': 'disgust', 'score': 0.06771168857812881},
 {'label': 'fear', 'score': 0.7942874431610107},
 {'label': 'joy', 'score': 0.012419751845300198},
 {'label': 'neutral', 'score': 0.03175150230526924},
 {'label': 'sadness', 'score': 0.0398513525724411},
 {'label': 'surprise', 'score': 0.03223242983222008}]

In [7]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

# creates a dictionary for each description containing the maximumm probability for each emotion
def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [8]:
# test for the first 10 books
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [9]:
emotion_scores

{'anger': [np.float64(0.4011019170284271),
  np.float64(0.39138707518577576),
  np.float64(0.8682214617729187),
  np.float64(0.15240122377872467),
  np.float64(0.33948376774787903),
  np.float64(0.06413350999355316),
  np.float64(0.06413350999355316),
  np.float64(0.04855480045080185),
  np.float64(0.06413350999355316),
  np.float64(0.9279194474220276)],
 'disgust': [np.float64(0.1949133574962616),
  np.float64(0.10400652140378952),
  np.float64(0.3810557723045349),
  np.float64(0.10400652140378952),
  np.float64(0.10400652140378952),
  np.float64(0.10400652140378952),
  np.float64(0.10400652140378952),
  np.float64(0.1865655928850174),
  np.float64(0.10400652140378952),
  np.float64(0.506156861782074)],
 'fear': [np.float64(0.9510065317153931),
  np.float64(0.05136270076036453),
  np.float64(0.9343246817588806),
  np.float64(0.7027426958084106),
  np.float64(0.6079992055892944),
  np.float64(0.05136270076036453),
  np.float64(0.18338298797607422),
  np.float64(0.06624423712491989),
  

In [10]:
from tqdm import tqdm
import re

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = re.split(r'[.?!]+', books["description"][i])
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

  0%|          | 0/100629 [00:00<?, ?it/s]

  0%|          | 1/100629 [00:00<2:50:35,  9.83it/s]

  0%|          | 3/100629 [00:00<2:36:38, 10.71it/s]

  0%|          | 7/100629 [00:00<1:18:44, 21.30it/s]

  0%|          | 10/100629 [00:00<1:31:38, 18.30it/s]

...

100%|█████████▉| 100617/100629 [1:18:55<00:00, 22.62it/s]

100%|█████████▉| 100620/100629 [1:18:55<00:00, 21.97it/s]

100%|█████████▉| 100623/100629 [1:18:55<00:00, 19.81it/s]

100%|█████████▉| 100627/100629 [1:18:55<00:00, 23.26it/s]

100%|██████████| 100629/100629 [1:18:55<00:00, 21.25it/s]

In [11]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [12]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.401102,0.194913,0.951007,0.069741,0.816087,0.305637,0.362398,9781411668980
1,0.391387,0.104007,0.051363,0.307681,0.549477,0.111690,0.150324,9780226575087
2,0.868221,0.381056,0.934325,0.276919,0.939046,0.640672,0.644575,9780062467874
3,0.152401,0.104007,0.702743,0.040564,0.874593,0.111690,0.078765,9780062930484
4,0.140983,0.104007,0.737003,0.258215,0.944252,0.111690,0.203355,9781632365804
...,...,...,...,...,...,...,...,...
100624,0.095671,0.104007,0.227203,0.066592,0.914617,0.111690,0.385701,9781925704259
100625,0.064134,0.104007,0.051363,0.226735,0.810443,0.111690,0.083311,9780642279606
100626,0.145456,0.338897,0.038927,0.612713,0.555899,0.283070,0.188844,9781988254685
100627,0.079136,0.104007,0.119965,0.572331,0.895729,0.111690,0.125690,9788198859686


In [13]:
# add the maximum score for each emotion to the book vectors
books = pd.merge(books, emotions_df, on = "isbn13")

In [14]:
books

,isbn13,title,authors,categories,description,thumbnail,published_year,average_rating,num_pages,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9781411668980,"The Galaxii Series: Book 1 ""Blachart""",Christina Engela,"Fiction, romance, general",Action! Adventure! Space Opera! Life hardly ev...,https://covers.openlibrary.org/b/id/14313750-L...,2018,NaN,280.0,"The Galaxii Series: Book 1 ""Blachart""",9781411668980 Action! Adventure! Space Opera! ...,Fiction,0.401102,0.194913,0.951007,0.069741,0.816087,0.305637,0.362398
1,9780226575087,Lost Mars,Michael Ashley,"Fiction;Science Fiction;Fiction, science ficti...",Ten short stories from the golden age of scien...,https://covers.openlibrary.org/b/id/13133252-L...,2018,NaN,302.0,Lost Mars: stories from the golden age of the ...,9780226575087 Ten short stories from the golde...,Fiction,0.391387,0.104007,0.051363,0.307681,0.549477,0.111690,0.150324
2,9780062467874,Villain,Michael Grant,Juvenile fiction;Fiction;Supernatural;Horror s...,MONSTER. VILLAIN. HERO. WHICH SUPERCREATURE WI...,https://covers.openlibrary.org/b/id/8814378-L.jpg,2018,NaN,324.0,Villain,9780062467874 MONSTER. VILLAIN. HERO. WHICH SU...,Children's,0.868221,0.381056,0.934325,0.276919,0.939046,0.640672,0.644575
3,9780062930484,The ABC Murders,Agatha Christie,Fiction;Mystery;Agatha Christie;Hercule Poirot...,"There's a serial killer on the loose, bent on ...",https://covers.openlibrary.org/b/id/-1-L.jpg,2019,NaN,272.0,The ABC Murders: A Hercule Poirot Mystery,9780062930484 There's a serial killer on the l...,Fiction,0.152401,0.104007,0.702743,0.040564,0.874593,0.111690,0.078765
4,9781632365804,Welcome to the ballroom,Tomo Takeuchi,Competitions;Ballroom dancing;Dance;Ballroom d...,"""Through sheer force of will, Tatara and China...",NaN,2018,NaN,NaN,Welcome to the ballroom,"9781632365804 ""Through sheer force of will, Ta...",Fiction,0.140983,0.104007,0.737003,0.258215,0.944252,0.111690,0.203355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100624,9781925704259,Dizzy limits,Noëlle Janaczewska,Australian Creative nonfiction,When conventional approaches to writing about ...,NaN,2020,NaN,440.0,Dizzy limits: recent experiments in Australian...,9781925704259 When conventional approaches to ...,Nonfiction,0.095671,0.104007,0.227203,0.066592,0.914617,0.111690,0.385701
100625,9780642279606,Flight of the Budgerigar,Penny Olsen,Budgerigar;History,Taking the reader from the Dreaming to the col...,NaN,2021,NaN,251.0,Flight of the Budgerigar: an illustrated history,9780642279606 Taking the reader from the Dream...,Nonfiction,0.064134,0.104007,0.051363,0.226735,0.810443,0.111690,0.083311
100626,9781988254685,Walls of the cave,Syr Ruus,Families;Fiction;Orphans;Loneliness,"""A writer of unknown gender, an orphan, brough...",NaN,2019,NaN,103.0,Walls of the cave,"9781988254685 ""A writer of unknown gender, an ...",Fiction,0.145456,0.338897,0.038927,0.612713,0.555899,0.283070,0.188844
100627,9788198859686,The Struggle for Europe,NaN,History;Biographies;Memoirs,"First published in 1952, ‘The Struggle for Eur...",https://covers.openlibrary.org/b/id/15219838-L...,2025,NaN,NaN,The Struggle for Europe,"9788198859686 First published in 1952, ‘The St...",Nonfiction,0.079136,0.104007,0.119965,0.572331,0.895729,0.111690,0.125690


In [15]:
books.to_csv("../data/books_with_emotions.csv", index = False)